# 11 — Kherson pilot: SWOT, ICESat-2 and the gauge

**Scientific question.** How do SWOT and ICESat-2 compare with the Kherson gauge
over the six downloaded pre-breach dates?

**Inputs** 6 PIXC granules, gauge 80805 daily series, ATL13 pass levels.
**Outputs** `Fig05_*`, `Fig07_*`, `Fig08_*` figure data.
**Statistical unit** SWOT: one overpass. ICESat-2: date × RGT. Bootstrap seed 42.

> **Caveat established in notebook 13:** the ICESat-2 pass-levels used here are an
> AOI-scale aggregate a median 29.8 km from the gauge, while SWOT is within 1 km.
> The gauge-mediated cross-sensor difference is therefore **confounded by the
> channel gradient**. Use notebook 13 for the like-for-like number.

In [1]:
# --- provenance: environment, git SHA and input hashes -----------------------
import hashlib, json, subprocess, sys, platform
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd, geopandas as gpd, pyproj, rasterio, matplotlib
from swot_dnipro import config as CFG

np.random.seed(CFG.SEED)

def sha256(p, cap=64 * 1024 * 1024):
    p = Path(p)
    if not p.exists() or p.stat().st_size > cap:
        return f"(skipped: {'missing' if not p.exists() else 'too large'})"
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()[:16]

def git_sha():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       cwd=CFG.ROOT, text=True).strip()
    except Exception:
        return "(not a git checkout)"

print("python      ", platform.python_version())
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("geopandas   ", gpd.__version__)
print("matplotlib  ", matplotlib.__version__)
print("pyproj/PROJ ", pyproj.__version__, "/", pyproj.proj_version_str)
print("rasterio/GDAL", rasterio.__version__, "/", rasterio.__gdal_version__)
print("git SHA     ", git_sha())
print("seed        ", CFG.SEED)
print()
for p in [CFG.EGG2015_TIF, CFG.UA2019Z_ASC, CFG.CORRECTOR_BY_STATION,
          CFG.KHERSON_GAUGE_PARQUET, CFG.KHERSON_ATL13]:
    print(f"  {sha256(p)}  {Path(p).name}")


python       3.12.3
numpy        2.5.2
pandas       3.0.5
geopandas    1.1.4
matplotlib   3.11.1
pyproj/PROJ  3.7.2 / 9.5.1
rasterio/GDAL 1.5.1 / 3.12.4
git SHA      7d9580b
seed         42

  (skipped: too large)  egg_2015.tif
  297440243e2ed8d6  ua_2019z.asc
  92d9fd71a1c8da39  egg2015_to_evrf2019_by_station.csv
  3380b29a0d68016b  80805_yearbook.parquet
  1deb7b5cfd362bd7  kherson_atl13_pass_levels.parquet


In [2]:
import subprocess
subprocess.run([sys.executable, "../scripts/make_figure_data.py"], check=True)

Building figure data ...
  [fig03] n=1023 nodes; A median=-0.0010 m
  [fig04] regional median -0.1695 -> -0.1321 m
  [kherson] SWOT n=6 dates, ICESat n=7 date×RGT, Δ_sensor V2=-0.1965 m, CI overlap=True
  [fig09] 5 mask strategies
  [fig12] 57 SWOT dates, 6 downloaded, 4 with same-day ICESat
  [fig06] Δt(SWOT−ICESat) = 14.22 h; accepted px = 145763
Done -> /home/niko/projects/SWOT-DNIPRO/outputs/figure_data


CompletedProcess(args=['/home/niko/projects/icesat2-atl13-kakhovka/.venv/bin/python3', '../scripts/make_figure_data.py'], returncode=0)

In [3]:
sw = pd.read_csv(CFG.FIGDATA / "Fig05_kherson_timeseries_swot.csv", parse_dates=["date", "swot_utc"])
ic = pd.read_csv(CFG.FIGDATA / "Fig05_kherson_timeseries_icesat.csv", parse_dates=["date", "icesat_utc"])
display(sw[["date", "swot_utc", "n_px", "H_SWOT_EGG2015_m", "nmad_m", "stage_bs77_m",
            "H_gauge_EVRF2019_m", "c_SWOT_m"]])
display(ic[["date", "icesat_utc", "rgt", "n_points", "H_tidefree_m", "H_meantide_m",
            "dist_to_gauge_km", "qc_flag"]])

,date,swot_utc,n_px,H_SWOT_EGG2015_m,nmad_m,stage_bs77_m,H_gauge_EVRF2019_m,c_SWOT_m
0,2023-04-05,2023-04-05 21:26:15.859364,1799,0.318898,0.174159,0.20,0.407787,0.088889
1,2023-04-15,2023-04-15 19:52:36.759107,1930,0.443438,0.150816,0.20,0.407787,-0.035652
2,2023-04-25,2023-04-25 18:18:55.536780,1553,0.505563,0.116648,0.34,0.547787,0.042223
3,2023-05-07,2023-05-07 16:26:28.839838,1785,0.529452,0.124537,0.29,0.497787,-0.031666
4,2023-05-23,2023-05-23 13:56:33.691009,1661,0.665318,0.120853,0.39,0.597787,-0.067531
5,2023-06-05,2023-06-05 11:54:45.729331,1616,0.526256,0.112708,0.32,0.527787,0.001530


,date,icesat_utc,rgt,n_points,H_tidefree_m,H_meantide_m,dist_to_gauge_km,qc_flag
0,2023-01-04,2023-01-04 11:33:03.517707264,225,1442,0.728648,0.692947,40.911689,ok
1,2023-02-12,2023-02-12 21:58:11.044438272,829,2351,0.334422,0.298887,19.132370,ok
2,2023-03-07,2023-03-07 08:36:50.497733376,1173,1964,0.428947,0.393371,21.365404,ok
3,2023-03-13,2023-03-13 20:34:11.968166912,1269,1632,0.641379,0.605690,40.011475,ok
4,2023-04-05,2023-04-05 07:12:58.836504576,225,718,0.655024,0.619365,37.042390,ok
5,2023-04-09,2023-04-09 07:04:29.656173824,289,21,0.281350,0.246408,22.564906,ok
6,2023-05-04,2023-05-04 05:48:53.829602560,669,1398,1.712504,1.676747,60.084654,BEAM_SPREAD_GT_1M
7,2023-05-14,2023-05-14 17:37:48.706915328,829,3305,0.914914,0.879386,18.995492,ok


## Residual statistics and harmonisation variants

In [4]:
display(pd.read_csv(CFG.FIGDATA / "Fig07_sensor_residuals.csv"))
display(pd.read_csv(CFG.FIGDATA / "Fig08_harmonisation_variants.csv"))

,sensor,statistic,median_c_m,nmad_m,ci95_low_m,ci95_high_m,n,independent_unit
0,SWOT (PIXC),"median (satellite − gauge, sign-flipped c)",-0.015068,0.054150,-0.051591,0.065556,6,SWOT overpasses
1,ICESat-2 (ATL13),"median (satellite − gauge, sign-flipped c)",-0.211579,0.118638,-0.285161,-0.005584,7,date x RGT
2,SWOT (PIXC),median R = H_sat − stage,0.222854,0.054150,0.142231,0.259378,6,SWOT overpasses
3,ICESat-2 (ATL13),median R = H_sat − stage,0.419365,0.118638,0.213371,0.492947,7,date x RGT


,variant,description,assessment,median_R_SWOT_m,R_SWOT_ci95_low,R_SWOT_ci95_high,nmad_SWOT_m,n_SWOT,median_R_ICESat_m,R_ICESat_ci95_low,R_ICESat_ci95_high,nmad_ICESat_m,n_ICESat,delta_sensor_m,ci_overlap
0,V0,SWOT raw height vs ATL13 tide-free,physically inconsistent (time-variable tides),0.288340,0.181697,0.460998,0.158108,6,0.455024,0.248947,0.528648,0.118446,7,-0.166684,True
1,V1,SWOT tide-corrected vs ATL13 tide-free,incomplete permanent-tide harmonisation,0.222854,0.142231,0.259378,0.054150,6,0.455024,0.248947,0.528648,0.118446,7,-0.232169,True
2,V2,+ ATL13 → mean/zero-tide crust,preferred branch — fully consistent with EGG2015,0.222854,0.142231,0.259378,0.054150,6,0.419365,0.213371,0.492947,0.118638,7,-0.196511,True
